# 🦖 Dino Shred — Build a Chrome Dino Clone in Pygame

### What you'll learn

| # | Concept | Game Dev Principle |
|---|---------|-------------------|
| 1 | **Game Loop** | INPUT → UPDATE → RENDER |
| 2 | **State Machine** | START / PLAYING / GAME_OVER |
| 3 | **Input Abstraction** | Actions, not raw keys |
| 4 | **Gravity & Jump Physics** | Velocity + acceleration |
| 5 | **AABB Collision** | Rectangle overlap detection |
| 6 | **Progressive Difficulty** | Speed ramps with score |
| 7 | **Object Management** | Spawn, cull, reuse |

Every section below has a **markdown cell** explaining the _why_, then a **code cell** you can run.

---
## Part 0: Imports & Constants

**Why constants?** Tweaking numbers in one place changes the entire game feel. Put every "magic number" in ALL_CAPS at the top.

In [ ]:
import pygame
import random
import sys

# ── Display ──────────────────────────────────────────
SCREEN_W = 800
SCREEN_H = 400
FPS = 60

# ── Colours ──────────────────────────────────────────
WHITE = (255, 255, 255)
BLACK = (30, 30, 30)
GREY  = (83, 83, 83)
DARK  = (50, 50, 50)

# ── Physics ──────────────────────────────────────────
GROUND_Y = 310          # y-coordinate of the horizon
GRAVITY  = 0.8          # pixels/frame² acceleration
JUMP_VEL = -14.0        # initial y-velocity (negative = up)

# ── Difficulty ───────────────────────────────────────
BASE_SPEED = 6.0        # starting scroll speed
MAX_SPEED  = 14.0       # never go faster than this
SPEED_UP_PER_POINT = 0.01

OBSTACLE_SPAWN_MIN = 50
OBSTACLE_SPAWN_MAX = 120

print("✅ Constants loaded — ready to go!")

---
## Part 1: The Dino

### What's happening?

- The dino stays at a **fixed x** (80 px from left). The world scrolls past it.
- **Gravity** pulls `vy` down each frame; **jumping** sets `vy` to a big negative number.
- `on_ground` is a flag — you can only jump when it's `True`.
- The `rect` property returns a slightly **padded hitbox** so collisions feel fair.

> **Core Principle #2 — State Machine:** The dino has three states: RUNNING, JUMPING, and DUCKING. We track them with simple booleans (`on_ground`, `ducking`).

In [ ]:
class Dino:
    """Player character — simple rectangle art, no sprites needed."""
    WIDTH  = 44
    HEIGHT = 48
    DUCK_HEIGHT = 28  # shorter when ducking
    X = 80            # fixed horizontal position

    def __init__(self):
        self.x = self.X
        self.y = GROUND_Y - self.HEIGHT
        self.vy = 0.0       # vertical velocity (pixels/frame)
        self.ducking = False
        self.on_ground = True

    # ── Actions (input abstraction) ──────────────────────
    def jump(self):
        """Only jump from the ground — no double jumps."""
        if self.on_ground:
            self.vy = JUMP_VEL
            self.on_ground = False

    def duck(self, is_ducking: bool):
        self.ducking = is_ducking

    # ── Physics (core principle: fixed timestep via FPS cap) ──
    def update(self):
        if not self.on_ground:
            self.vy += GRAVITY          # accelerate downward
            self.y += self.vy            # apply velocity

            # Hit the ground?
            floor_y = GROUND_Y - (self.DUCK_HEIGHT if self.ducking else self.HEIGHT)
            if self.y >= floor_y:
                self.y = floor_y
                self.vy = 0
                self.on_ground = True

    # ── Rendering ───────────────────────────────────────
    def draw(self, screen: pygame.Surface):
        h = self.DUCK_HEIGHT if self.ducking else self.HEIGHT
        body = pygame.Rect(self.x, self.y, self.WIDTH, h)
        pygame.draw.rect(screen, DARK, body)

        # Eye
        eye_x = self.x + (30 if not self.ducking else 10)
        pygame.draw.circle(screen, WHITE, (eye_x, self.y + 12), 4)

        # Animated legs (swap every 150 ms)
        leg_phase = (pygame.time.get_ticks() // 150) % 2
        offset = 3 if leg_phase == 0 else -3
        leg_y = self.y + h
        pygame.draw.rect(screen, DARK, (self.x + 8,  leg_y, 6, 6 + offset))
        pygame.draw.rect(screen, DARK, (self.x + 20, leg_y, 6, 6 - offset))

    # ── Collision hitbox (AABB — core principle #6) ────
    @property
    def rect(self) -> pygame.Rect:
        h = self.DUCK_HEIGHT if self.ducking else self.HEIGHT
        return pygame.Rect(self.x + 4, self.y + 4,
                           self.WIDTH - 8, h - 4)

print("✅ Dino class defined")

---
## Part 2: Obstacles

### Design decisions:

- **4 variants:** small cactus, tall cactus, bird-high, bird-low.
- Cacti sit **on the ground**; birds float **above** it (so you must either jump or duck).
- Obstacles spawn at `SCREEN_W` (right edge) and scroll left. When they leave the left edge we **cull** them.

> **Core Principle #2 — Object Pooling:** We don't `del` obstacles; we just drop them from the list. Python's GC handles the rest at this scale.

In [ ]:
class Obstacle:
    """Scrolling obstacle — cacti and birds."""

    # (width, height) per variant
    SIZES = {
        0: (20, 40),   # small cactus
        1: (24, 54),   # tall cactus
        2: (42, 20),   # bird (high)
        3: (42, 20),   # bird (low)
    }
    BIRD_Y_OFFSETS = {2: 100, 3: 60}

    def __init__(self, variant: int):
        self.variant = variant
        w, h = self.SIZES[variant]
        self.width, self.height = w, h

        # Start at right edge
        self.x = float(SCREEN_W)
        if variant in (0, 1):      # cactus — on ground
            self.y = GROUND_Y - h
        else:                       # bird — float above ground
            self.y = GROUND_Y - h - self.BIRD_Y_OFFSETS[variant]

    def update(self, speed: float):
        self.x -= speed

    def is_off_screen(self) -> bool:
        return self.x + self.width < 0

    def draw(self, screen: pygame.Surface):
        rect = pygame.Rect(self.x, self.y, self.width, self.height)
        color = GREY if self.variant in (0, 1) else (120, 120, 120)
        pygame.draw.rect(screen, color, rect)

    @property
    def rect(self) -> pygame.Rect:
        return pygame.Rect(self.x + 2, self.y + 2,
                           self.width - 4, self.height - 4)

print("✅ Obstacle class defined")

---
## Part 3: Scrolling Ground

The ground doesn't actually move — we redraw a dashed line with a **shifting offset** (`scroll`). This is an old-school trick: cheat the eye, not the CPU.

In [ ]:
class Ground:
    """Scrolling dashed line — pure illusion of motion."""
    DASH_LENGTH = 12
    DASH_GAP    = 6
    DASH_Y      = GROUND_Y + 2

    def __init__(self):
        self.scroll = 0.0

    def update(self, speed: float):
        period = self.DASH_LENGTH + self.DASH_GAP
        self.scroll = (self.scroll + speed) % period

    def draw(self, screen: pygame.Surface):
        # Solid horizon line
        pygame.draw.line(screen, DARK, (0, GROUND_Y),
                         (SCREEN_W, GROUND_Y), 2)
        # Scrolling dashes below it
        period = self.DASH_LENGTH + self.DASH_GAP
        x = -self.scroll
        while x < SCREEN_W:
            pygame.draw.line(screen, DARK,
                             (x, self.DASH_Y),
                             (x + self.DASH_LENGTH, self.DASH_Y), 2)
            x += period

print("✅ Ground class defined")

---
## Part 4: The State Machine

### This is the most important architectural pattern in game dev.

Three states:

```
  START ──(press SPACE)──▶ PLAYING ──(hit obstacle)──▶ GAME_OVER
    ▲                                                   │
    └───────────────(press SPACE)───────────────────────┘
```

The `update()` and `draw()` methods **check `self.state`** and behave differently. No nested if-else spaghetti.

> **Core Principle #5 — Progressive Difficulty:** `self.speed` increases by `SPEED_UP_PER_POINT` each point. Spawn windows tighten too. The game gets harder the longer you survive.

In [ ]:
class Game:
    """Top-level game state machine."""

    def __init__(self):
        self.state = "START"
        self.dino = Dino()
        self.ground = Ground()
        self.obstacles: list[Obstacle] = []
        self.score = 0
        self.high_score = 0
        self.speed = BASE_SPEED
        self.spawn_timer = 0
        self.next_spawn_in = random.randint(OBSTACLE_SPAWN_MIN,
                                            OBSTACLE_SPAWN_MAX)
        self.font = pygame.font.Font(None, 24)
        self.big_font = pygame.font.Font(None, 48)

    # ── Helpers ──────────────────────────────────────────
    def _reset(self):
        self.dino = Dino()
        self.obstacles.clear()
        self.score = 0
        self.speed = BASE_SPEED
        self.spawn_timer = 0
        self.next_spawn_in = random.randint(OBSTACLE_SPAWN_MIN,
                                            OBSTACLE_SPAWN_MAX)

    def _spawn_obstacle(self):
        """Pick a random variant. Early game = only cacti."""
        if self.score < 200:
            variant = random.randint(0, 1)
        elif self.score < 500:
            variant = random.choices([0, 1, 2], weights=[3, 3, 1])[0]
        else:
            variant = random.choices([0, 1, 2, 3], weights=[2, 2, 2, 1])[0]
        self.obstacles.append(Obstacle(variant))

    # ── Per-frame update ─────────────────────────────────
    def update(self):
        if self.state != "PLAYING":
            return

        # Increase speed with score
        self.speed = min(MAX_SPEED, BASE_SPEED + self.score * SPEED_UP_PER_POINT)

        self.dino.update()
        self.ground.update(self.speed)

        # Move obstacles & cull off-screen
        for obs in self.obstacles:
            obs.update(self.speed)
        self.obstacles = [o for o in self.obstacles if not o.is_off_screen()]

        # Spawn logic
        self.spawn_timer += 1
        if self.spawn_timer >= self.next_spawn_in:
            self._spawn_obstacle()
            self.spawn_timer = 0
            spawn_min = max(25, OBSTACLE_SPAWN_MIN - int(self.speed - BASE_SPEED) * 3)
            spawn_max = max(50, OBSTACLE_SPAWN_MAX - int(self.speed - BASE_SPEED) * 6)
            self.next_spawn_in = random.randint(spawn_min, spawn_max)

        self.score += 1

        # ── Collision (AABB) ─────────────────────────────
        dino_rect = self.dino.rect
        for obs in self.obstacles:
            if dino_rect.colliderect(obs.rect):
                self.state = "GAME_OVER"
                if self.score > self.high_score:
                    self.high_score = self.score
                return

    # ── Rendering ───────────────────────────────────────
    def draw(self, screen: pygame.Surface):
        screen.fill(WHITE)
        self.ground.draw(screen)
        self.dino.draw(screen)
        for obs in self.obstacles:
            obs.draw(screen)

        # HUD
        score_surf = self.font.render(f"Score: {self.score:05d}", True, GREY)
        screen.blit(score_surf, (SCREEN_W - 150, 20))
        hi_surf = self.font.render(f"HI: {self.high_score:05d}", True, GREY)
        screen.blit(hi_surf, (SCREEN_W - 300, 20))

        # Overlays
        if self.state == "START":
            self._draw_overlay(screen, "DINO SHRED", "Press SPACE or ↑ to start")
        elif self.state == "GAME_OVER":
            self._draw_overlay(screen, "GAME OVER", "Press SPACE or ↑ to retry")

    def _draw_overlay(self, screen, title, subtitle):
        s = pygame.Surface((SCREEN_W, SCREEN_H), pygame.SRCALPHA)
        s.fill((255, 255, 255, 120))
        screen.blit(s, (0, 0))
        t = self.big_font.render(title, True, DARK)
        screen.blit(t, (SCREEN_W // 2 - t.get_width() // 2, SCREEN_H // 2 - 80))
        st = self.font.render(subtitle, True, GREY)
        screen.blit(st, (SCREEN_W // 2 - st.get_width() // 2, SCREEN_H // 2 - 30))

    # ── Input (core principle #3: actions, not keys) ────
    def handle_event(self, event: pygame.event.Event):
        if event.type == pygame.KEYDOWN:
            # SPACE / UP → start, restart, or jump depending on state
            if event.key in (pygame.K_SPACE, pygame.K_UP):
                if self.state == "START":
                    self.state = "PLAYING"
                elif self.state == "GAME_OVER":
                    self._reset()
                    self.state = "PLAYING"
                elif self.state == "PLAYING":
                    self.dino.jump()
            # DOWN → duck
            if event.key == pygame.K_DOWN and self.state == "PLAYING":
                self.dino.duck(True)

        elif event.type == pygame.KEYUP:
            if event.key == pygame.K_DOWN and self.state == "PLAYING":
                self.dino.duck(False)

print("✅ Game class defined")

---
## Part 5: The Game Loop

### Every frame follows this exact order:

```
┌─────────────────────────────────┐
│  INPUT   │  poll events, call   │
│          │  handle_event()      │
├──────────┼──────────────────────┤
│  UPDATE  │  physics, spawn,     │
│          │  cull, collision     │
├──────────┼──────────────────────┤
│  RENDER  │  fill, draw, HUD,    │
│          │  display.flip()      │
├──────────┼──────────────────────┤
│  WAIT    │  clock.tick(60)      │
│          │  → caps at 60 FPS    │
└─────────────────────────────────┘
```

**Why `clock.tick(FPS)`?** Without it, the game runs as fast as the CPU allows — unplayably fast. Ticking caps updates to 60/s, making physics consistent across machines.

> **⚠️ Heads-up:** Running this cell opens a pygame window. Close it or press **Cmd+.** in Jupyter to stop the kernel.

In [ ]:
def main():
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_W, SCREEN_H))
    pygame.display.set_caption("Dino Shred 🦖")
    clock = pygame.time.Clock()

    game = Game()

    while True:
        # ── INPUT ───────────────────────────────────
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return  # exits main(), returns to notebook
            game.handle_event(event)

        # ── UPDATE ──────────────────────────────────
        game.update()

        # ── RENDER ──────────────────────────────────
        game.draw(screen)
        pygame.display.flip()
        clock.tick(FPS)


# Run it!
main()
print("👋 Game window closed — back in the notebook.")

---
## Part 6: What Next? — Exercises

Each of these builds on what you now understand. Try them in order:

| Level | Task | Hint |
|-------|------|------|
| 🟢 | Add a **double-jump** | Add a `jumps_left` counter, reset on landing |
| 🟢 | Add **sound effects** | `pygame.mixer.Sound("jump.wav").play()` |
| 🟡 | Replace rectangles with **sprites** | `pygame.image.load("dino.png")` + `screen.blit()` |
| 🟡 | Add **clouds** scrolling at half speed | Same pattern as Ground, different layer |
| 🔴 | Add a **start menu** and **pause** | New state `PAUSED`, toggle with `P` key |
| 🔴 | Make the dino **crouch-slide** while ducking | Reduce `DUCK_HEIGHT` over time, then stand up |
| 🔴 | Add **particles** when dino lands | Spawn 5 small rects that fade out in 300 ms |

### Key takeaways

1. **The game loop** is input → update → render. Always.
2. **State machines** prevent spaghetti code. Every `if self.state == ...` is a clean branch.
3. **Input abstraction** means mapping keys to *actions*, not checking raw keycodes everywhere.
4. **AABB collision** (`rect.colliderect()`) handles 90% of 2D collision needs.
5. **Progressive difficulty** is just `speed = base + score * factor` — simple but effective.
6. **Constants at the top** let you tune the game in seconds without hunting through code.

---

### 🦖 Go shred!